In [1]:
import os
import sys
import time

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

26/09/16 22:12:07 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 172.27.138.95 instead (on interface en0)
26/09/16 22:12:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 22:12:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


### Load tables

In [2]:
from deltalake import DeltaTable
from pyspark.sql import functions as F
from src.analytics.queries import (
    ALL_QUERIES,
    ALL_QUERIES_WITHOUT_FILTER,
    QUERY_MONTHLY_DEMAND_PER_ZONE,
    QUERY_MONTHLY_TRENDS,
    QUERY_ZONES_MOST_WEATHER_SENSITIVE,
    WEATHER_CONDITION_CASE,
    QUERY_AIR_QUALITY_VS_DEMAND,
)

GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "gold", "integrated_taxi_trips")
TRIPDATA_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "silver", "tripdata")
WEATHER_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "silver", "weather")

def read_delta(path):
    files = DeltaTable(path).file_uris()
    return spark.read.option("basePath", path).parquet(*files)

gold = read_delta(GOLD_PATH).filter(F.col("is_valid"))
gold.createOrReplaceTempView("integrated_taxi_trips")

trips = read_delta(TRIPDATA_PATH).filter(F.col("is_valid"))
weather = read_delta(WEATHER_PATH).filter(F.col("is_valid"))
trips.createOrReplaceTempView("tripdata")
weather.createOrReplaceTempView("weather")

# Warm up the session once - the FIRST query in a fresh session pays a one-time
# JVM/codegen warmup cost that would otherwise bias every "before" measurement
# below unfairly, regardless of which optimization is being tested.
spark.sql("SELECT COUNT(*) FROM integrated_taxi_trips").collect()
print("Setup complete.")

26/09/16 22:12:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Setup complete.


## Optimization

### Partition Pruning with Q1 & Q6

In [29]:
# Partition pruning experiment for Q1 and Q6 - Jan 2024
import time

# Q1 baseline: same Jan scope, without partition-column filtering
q1_baseline = """
SELECT
    pickup_zone,
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count
FROM integrated_taxi_trips
WHERE event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_zone, pickup_year, pickup_month
ORDER BY pickup_year, pickup_month, trip_count DESC
"""

# Q1 optimized: same Jan scope + partition filters
q1_filtered = """
SELECT
    pickup_zone,
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count
FROM integrated_taxi_trips
WHERE pickup_year = 2024
  AND pickup_month = 1
  AND event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_zone, pickup_year, pickup_month
ORDER BY pickup_year, pickup_month, trip_count DESC
"""

# Q6 baseline
q6_baseline = """
SELECT
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM integrated_taxi_trips
WHERE event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_year, pickup_month
ORDER BY pickup_year, pickup_month
"""

# Q6 optimized with partition pruning
q6_filtered = """
SELECT
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM integrated_taxi_trips
WHERE pickup_year = 2024
  AND pickup_month = 1
  AND event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_year, pickup_month
ORDER BY pickup_year, pickup_month
"""

def same_results(df1, df2):
    return (
        df1.exceptAll(df2).limit(1).count() == 0
        and df2.exceptAll(df1).limit(1).count() == 0
    )

for label, baseline, filtered in [
    ("Q1", q1_baseline, q1_filtered),
    ("Q6", q6_baseline, q6_filtered),
]:
    # Baseline job label for Spark UI
    spark.sparkContext.setJobGroup(
        f"{label}_PARTITION_BASELINE",
        f"{label} - Partition pruning baseline"
    )

    t0 = time.perf_counter()
    baseline_df = spark.sql(baseline)
    baseline_df.collect()
    t_baseline = time.perf_counter() - t0

    spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
    spark.sparkContext.setLocalProperty("spark.job.description", None)

    # Optimized job label for Spark UI
    spark.sparkContext.setJobGroup(
        f"{label}_PARTITION_OPTIMIZED",
        f"{label} - Partition pruning optimized"
    )

    t0 = time.perf_counter()
    filtered_df = spark.sql(filtered)
    filtered_df.collect()
    t_filtered = time.perf_counter() - t0

    spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
    spark.sparkContext.setLocalProperty("spark.job.description", None)

    print(
        f"{label}: baseline={t_baseline:.3f}s  "
        f"partition-pruned={t_filtered:.3f}s  "
        f"speedup={t_baseline / t_filtered:.2f}x"
    )
    print("Same result:", same_results(baseline_df, filtered_df))
    print()

print("=== Q6 BASELINE PLAN ===")
spark.sql(q6_baseline).explain("formatted")

print("\n=== Q6 PARTITION-PRUNED PLAN ===")
spark.sql(q6_filtered).explain("formatted")

Q1: baseline=0.291s  partition-pruned=0.200s  speedup=1.45x
Same result: True

Q6: baseline=0.148s  partition-pruned=0.071s  speedup=2.07x
Same result: True

=== Q6 BASELINE PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [event_date#0, trip_distance#8, fare_amount#12, is_valid#28, pickup_year#41, pickup_month#42]
Batched: true
Location: InMemoryFileIndex [file:/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/delta/gold/integrated_taxi_trips/pickup_year=2024/pickup_month=1/part-00000-14624fe6-0b4e-4a6d-9fde-2f0197308dec-c000.snappy.parquet, ... 9 entries]
PushedFilters: [IsNotNull(is_valid), IsNotNull(event_date), EqualTo(is_valid,true), GreaterThanOrEqual(event_date,2024-01-01), LessT

In [52]:
# Partition pruning experiment for Q1 and Q6 - Jan 2024
import time

# Q1 baseline: same Jan scope, without partition-column filtering
q1_baseline = """
SELECT
    pickup_zone,
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count
FROM integrated_taxi_trips
WHERE event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_zone, pickup_year, pickup_month
ORDER BY pickup_year, pickup_month, trip_count DESC
"""

# Q1 optimized: same Jan scope + partition filters
q1_filtered = """
SELECT
    pickup_zone,
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count
FROM integrated_taxi_trips
WHERE pickup_year = 2024
  AND pickup_month = 1
  AND event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_zone, pickup_year, pickup_month
ORDER BY pickup_year, pickup_month, trip_count DESC
"""

# Q6 baseline
q6_baseline = """
SELECT
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM integrated_taxi_trips
WHERE event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_year, pickup_month
ORDER BY pickup_year, pickup_month
"""

# Q6 optimized with partition pruning
q6_filtered = """
SELECT
    pickup_year,
    pickup_month,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM integrated_taxi_trips
WHERE pickup_year = 2024
  AND pickup_month = 1
  AND event_date >= DATE '2024-01-01'
  AND event_date <  DATE '2024-02-01'
GROUP BY pickup_year, pickup_month
ORDER BY pickup_year, pickup_month
"""

def same_results(df1, df2):
    return (
        df1.exceptAll(df2).limit(1).count() == 0
        and df2.exceptAll(df1).limit(1).count() == 0
    )

for label, baseline, filtered in [
    ("Q1", q1_baseline, q1_filtered),
    ("Q6", q6_baseline, q6_filtered),
]:
    # Baseline job label for Spark UI
    spark.sparkContext.setJobGroup(
        f"{label}_PARTITION_BASELINE",
        f"{label} - Partition pruning baseline"
    )

    t0 = time.perf_counter()
    baseline_df = spark.sql(baseline)
    baseline_df.collect()
    t_baseline = time.perf_counter() - t0

    spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
    spark.sparkContext.setLocalProperty("spark.job.description", None)

    # Optimized job label for Spark UI
    spark.sparkContext.setJobGroup(
        f"{label}_PARTITION_OPTIMIZED",
        f"{label} - Partition pruning optimized"
    )

    t0 = time.perf_counter()
    filtered_df = spark.sql(filtered)
    filtered_df.collect()
    t_filtered = time.perf_counter() - t0

    spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
    spark.sparkContext.setLocalProperty("spark.job.description", None)

    print(
        f"{label}: baseline={t_baseline:.3f}s  "
        f"partition-pruned={t_filtered:.3f}s  "
        f"speedup={t_baseline / t_filtered:.2f}x"
    )
    print("Same result:", same_results(baseline_df, filtered_df))
    print()

print("=== Q6 BASELINE PLAN ===")
spark.sql(q6_baseline).explain("formatted")

print("\n=== Q6 PARTITION-PRUNED PLAN ===")
spark.sql(q6_filtered).explain("formatted")

Q1: baseline=0.133s  partition-pruned=0.097s  speedup=1.37x
Same result: True

Q6: baseline=0.084s  partition-pruned=0.063s  speedup=1.34x
Same result: True

=== Q6 BASELINE PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [event_date#450, trip_distance#458, fare_amount#462, is_valid#478, pickup_year#491, pickup_month#492]
Batched: true
Location: InMemoryFileIndex [file:/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/delta/gold/integrated_taxi_trips/pickup_year=2024/pickup_month=1/part-00000-14624fe6-0b4e-4a6d-9fde-2f0197308dec-c000.snappy.parquet, ... 9 entries]
PushedFilters: [IsNotNull(is_valid), IsNotNull(event_date), EqualTo(is_valid,true), GreaterThanOrEqual(event_date,2024-01-01

### Broadcast Joins (Q2, live join)

In [53]:
# Broadcast join experiment for Q2
# Baseline: force a sort-merge join.
# Optimized: broadcast the small weather table to avoid shuffling both tables.

import time

# Baseline query - force sort-merge join
q2_baseline = f"""
SELECT /*+ MERGE(t, w) */
       {WEATHER_CONDITION_CASE.replace("weather_condition_code", "w.condition_code")} AS weather_condition,
       COUNT(*) AS trip_count,
       ROUND(AVG(t.trip_distance), 3) AS avg_distance
FROM tripdata t
JOIN weather w
    ON t.event_date = w.event_date
   AND t.event_hour = w.event_hour
GROUP BY 1
ORDER BY trip_count DESC
"""

# Optimized query - broadcast the small weather table
q2_optimized = f"""
SELECT /*+ BROADCAST(w) */
       {WEATHER_CONDITION_CASE.replace("weather_condition_code", "w.condition_code")} AS weather_condition,
       COUNT(*) AS trip_count,
       ROUND(AVG(t.trip_distance), 3) AS avg_distance
FROM tripdata t
JOIN weather w
    ON t.event_date = w.event_date
   AND t.event_hour = w.event_hour
GROUP BY 1
ORDER BY trip_count DESC
"""

baseline_times = []
optimized_times = []

for _ in range(3):
    t0 = time.perf_counter()
    spark.sparkContext.setJobGroup("Q2_BASELINE", "Q2 - Sort-merge join baseline")
    baseline_result = spark.sql(q2_baseline).collect()
    baseline_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    spark.sparkContext.setJobGroup("Q2_BROADCAST", "Q2 - Broadcast join optimized")
    optimized_result = spark.sql(q2_optimized).collect()
    optimized_times.append(time.perf_counter() - t0)

baseline_median = statistics.median(baseline_times)
optimized_median = statistics.median(optimized_times)

print(f"Baseline sort-merge join: median={baseline_median:.3f}s")
print(f"Optimized broadcast join: median={optimized_median:.3f}s")
print(f"Speedup: {baseline_median / optimized_median:.2f}x")

# Verify both queries return the same results
baseline_sorted = sorted(baseline_result, key=lambda r: r["weather_condition"])
optimized_sorted = sorted(optimized_result, key=lambda r: r["weather_condition"])

print("Same result:", baseline_sorted == optimized_sorted)

# Compare physical execution plans
print("\n=== BASELINE PLAN: SORT-MERGE JOIN ===")
spark.sql(q2_baseline).explain("formatted")

print("\n=== OPTIMIZED PLAN: BROADCAST JOIN ===")
spark.sql(q2_optimized).explain("formatted")

Baseline sort-merge join: median=0.851s
Optimized broadcast join: median=0.558s
Speedup: 1.53x
Same result: True

=== BASELINE PLAN: SORT-MERGE JOIN ===
== Physical Plan ==
AdaptiveSparkPlan (18)
+- Sort (17)
   +- Exchange (16)
      +- HashAggregate (15)
         +- Exchange (14)
            +- HashAggregate (13)
               +- Project (12)
                  +- SortMergeJoin Inner (11)
                     :- Sort (5)
                     :  +- Exchange (4)
                     :     +- Project (3)
                     :        +- Filter (2)
                     :           +- Scan parquet  (1)
                     +- Sort (10)
                        +- Exchange (9)
                           +- Project (8)
                              +- Filter (7)
                                 +- Scan parquet  (6)


(1) Scan parquet 
Output [6]: [trip_distance#541, event_date#557, event_hour#558, is_valid#565, pickup_year#566, pickup_month#567]
Batched: true
Location: InMemoryFileIndex [fil

### Adaptive Query Execution (Q4)

In [59]:
# Compare the same query with AQE disabled and enabled to measure its effect on performance.

import time
import statistics

# Warm up Spark once before benchmarking
spark.sql(QUERY_ZONES_MOST_WEATHER_SENSITIVE).collect()

aqe_results = {}

# Baseline: AQE disabled
spark.conf.set("spark.sql.adaptive.enabled", "false")

baseline_times = []
for _ in range(3):
    spark.sparkContext.setJobGroup(
        "Q4_AQE_DISABLED",
        "Q4 - AQE disabled"
    )
    
    t0 = time.perf_counter()
    baseline_result = spark.sql(QUERY_ZONES_MOST_WEATHER_SENSITIVE).collect()
    baseline_times.append(time.perf_counter() - t0)

baseline_median = statistics.median(baseline_times)
aqe_results["baseline"] = (baseline_median, baseline_result)

print(
    f"Baseline - AQE disabled: median={baseline_median:.3f}s "
    f"runs={[round(t, 3) for t in baseline_times]}"
)

# Optimized: AQE enabled
spark.conf.set("spark.sql.adaptive.enabled", "true")

optimized_times = []
for _ in range(3):
    spark.sparkContext.setJobGroup(
        "Q4_AQE_ENABLED",
        "Q4 - AQE enabled"
    )
    
    t0 = time.perf_counter()
    optimized_result = spark.sql(QUERY_ZONES_MOST_WEATHER_SENSITIVE).collect()
    optimized_times.append(time.perf_counter() - t0)

optimized_median = statistics.median(optimized_times)
aqe_results["optimized"] = (optimized_median, optimized_result)

print(
    f"Optimized - AQE enabled: median={optimized_median:.3f}s "
    f"runs={[round(t, 3) for t in optimized_times]}"
)

print(f"Speedup: {baseline_median / optimized_median:.2f}x")
print("Same result:", baseline_result == optimized_result)

print("\n=== BASELINE PLAN: AQE DISABLED ===")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.sql(QUERY_ZONES_MOST_WEATHER_SENSITIVE).explain("formatted")

print("\n=== OPTIMIZED PLAN: AQE ENABLED ===")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.sql(QUERY_ZONES_MOST_WEATHER_SENSITIVE).explain("formatted")

# Keep AQE enabled after the experiment
spark.conf.set("spark.sql.adaptive.enabled", "true")

Baseline - AQE disabled: median=0.163s runs=[0.228, 0.163, 0.158]
Optimized - AQE enabled: median=0.147s runs=[0.147, 0.153, 0.144]
Speedup: 1.11x
Same result: True

=== BASELINE PLAN: AQE DISABLED ===
== Physical Plan ==
TakeOrderedAndProject (14)
+- * Filter (13)
   +- * HashAggregate (12)
      +- Exchange (11)
         +- * HashAggregate (10)
            +- * HashAggregate (9)
               +- * HashAggregate (8)
                  +- * HashAggregate (7)
                     +- Exchange (6)
                        +- * HashAggregate (5)
                           +- * Project (4)
                              +- * Filter (3)
                                 +- * ColumnarToRow (2)
                                    +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [is_valid#478, pickup_zone#479, weather_condition_code#488, pickup_year#491, pickup_month#492]
Batched: true
Location: InMemoryFileIndex [file:/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mob

### AQE with Q3

In [38]:
# Compare the same query with AQE disabled and enabled to measure its effect on performance.

import time
import statistics

# Warm up Spark once before benchmarking
spark.sql(QUERY_AIR_QUALITY_VS_DEMAND).collect()

aqe_results = {}

# Baseline: AQE disabled
spark.conf.set("spark.sql.adaptive.enabled", "false")

baseline_times = []
for _ in range(3):
    spark.sparkContext.setJobGroup(
        "Q3_AQE_DISABLED",
        "Q3 - AQE disabled"
    )
    
    t0 = time.perf_counter()
    baseline_result = spark.sql(QUERY_AIR_QUALITY_VS_DEMAND).collect()
    baseline_times.append(time.perf_counter() - t0)

baseline_median = statistics.median(baseline_times)
aqe_results["baseline"] = (baseline_median, baseline_result)

print(
    f"Baseline - AQE disabled: median={baseline_median:.3f}s "
    f"runs={[round(t, 3) for t in baseline_times]}"
)

# Optimized: AQE enabled
spark.conf.set("spark.sql.adaptive.enabled", "true")

optimized_times = []
for _ in range(3):
    spark.sparkContext.setJobGroup(
        "Q3_AQE_ENABLED",
        "Q3 - AQE enabled"
    )
    
    t0 = time.perf_counter()
    optimized_result = spark.sql(QUERY_AIR_QUALITY_VS_DEMAND).collect()
    optimized_times.append(time.perf_counter() - t0)

optimized_median = statistics.median(optimized_times)
aqe_results["optimized"] = (optimized_median, optimized_result)

print(
    f"Optimized - AQE enabled: median={optimized_median:.3f}s "
    f"runs={[round(t, 3) for t in optimized_times]}"
)

print(f"Speedup: {baseline_median / optimized_median:.2f}x")
print("Same result:", baseline_result == optimized_result)

print("\n=== BASELINE PLAN: AQE DISABLED ===")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.sql(QUERY_AIR_QUALITY_VS_DEMAND).explain("formatted")

print("\n=== OPTIMIZED PLAN: AQE ENABLED ===")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.sql(QUERY_AIR_QUALITY_VS_DEMAND).explain("formatted")

# Keep AQE enabled after the experiment
spark.conf.set("spark.sql.adaptive.enabled", "true")

Baseline - AQE disabled: median=0.067s runs=[0.067, 0.075, 0.057]
Optimized - AQE enabled: median=0.059s runs=[0.06, 0.056, 0.059]
Speedup: 1.14x
Same result: True

=== BASELINE PLAN: AQE DISABLED ===
== Physical Plan ==
* Sort (9)
+- Exchange (8)
   +- * HashAggregate (7)
      +- Exchange (6)
         +- * HashAggregate (5)
            +- * Project (4)
               +- * Filter (3)
                  +- * ColumnarToRow (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [is_valid#28, air_quality_pm25#39, pickup_year#41, pickup_month#42]
Batched: true
Location: InMemoryFileIndex [file:/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/delta/gold/integrated_taxi_trips/pickup_year=2024/pickup_month=1/part-00000-14624fe6-0b4e-4a6d-9fde-2f0197308dec-c000.snappy.parquet, ... 9 entries]
PushedFilters: [IsNotNull(is_valid), EqualTo(is_valid,true), IsNotNull(air_quality_pm25)]
ReadSchema: struct<is_valid:boolean,air_qu

### Caching (all 6 queries)

In [42]:
import time
import statistics

gold_slim = gold.select(
    "pickup_zone", "pickup_borough", "dropoff_zone", "dropoff_borough",
    "pickup_year", "pickup_month",
    "event_timestamp", "event_date", "event_hour",
    "weather_condition_code", "air_quality_pm25",
    "trip_distance", "fare_amount", "trip_duration_minutes",
)

gold_slim.createOrReplaceTempView("integrated_taxi_trips")

RUNS = 3

# Warmup
spark.sql(list(ALL_QUERIES_WITHOUT_FILTER.values())[0]).collect()


# UNCACHED

uncached_times = {}
uncached_results = {}

for name, query in ALL_QUERIES_WITHOUT_FILTER.items():
    times = []

    for _ in range(RUNS):
        spark.sparkContext.setJobGroup(
            f"UNCACHED_{name}",
            f"Uncached - {name}"
        )

        t0 = time.perf_counter()
        result = spark.sql(query).collect()
        times.append(time.perf_counter() - t0)

    uncached_times[name] = statistics.median(times)
    uncached_results[name] = result

    print(
        f"{name} uncached: "
        f"median={uncached_times[name]:.3f}s "
        f"runs={[round(t, 3) for t in times]}"
    )

spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
spark.sparkContext.setLocalProperty("spark.job.description", None)


# Show execution plan for one representative uncached query
plan_name, plan_query = next(iter(ALL_QUERIES_WITHOUT_FILTER.items()))

print(f"\n=== UNCACHED PLAN: {plan_name} ===")
spark.sql(plan_query).explain("formatted")



# CACHE

gold_slim.cache()

spark.sparkContext.setJobGroup(
    "CACHE_MATERIALIZE",
    "Cache - Materialize gold_slim"
)

gold_slim.count()  # force cache materialization

spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
spark.sparkContext.setLocalProperty("spark.job.description", None)


cached_times = {}
cached_results = {}

for name, query in ALL_QUERIES_WITHOUT_FILTER.items():
    times = []

    for _ in range(RUNS):
        spark.sparkContext.setJobGroup(
            f"CACHED_{name}",
            f"Cached - {name}"
        )

        t0 = time.perf_counter()
        result = spark.sql(query).collect()
        times.append(time.perf_counter() - t0)

    cached_times[name] = statistics.median(times)
    cached_results[name] = result

    print(
        f"{name} cached: "
        f"median={cached_times[name]:.3f}s "
        f"runs={[round(t, 3) for t in times]}"
    )

spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
spark.sparkContext.setLocalProperty("spark.job.description", None)



# SUMMARY

print("\n=== CACHING BENCHMARK SUMMARY ===")

for name in ALL_QUERIES_WITHOUT_FILTER:
    same_result = (
        sorted(map(str, uncached_results[name]))
        == sorted(map(str, cached_results[name]))
    )

    print(
        f"{name}: "
        f"uncached={uncached_times[name]:.3f}s  "
        f"cached={cached_times[name]:.3f}s  "
        f"speedup={uncached_times[name] / cached_times[name]:.2f}x  "
        f"same_result={same_result}"
    )

total_uncached = sum(uncached_times.values())
total_cached = sum(cached_times.values())

print(
    f"\nAll queries: "
    f"uncached={total_uncached:.3f}s  "
    f"cached={total_cached:.3f}s  "
    f"speedup={total_uncached / total_cached:.2f}x"
)


# Show execution plan for same query after caching
print(f"\n=== CACHED PLAN: {plan_name} ===")
spark.sql(plan_query).explain("formatted")


# CLEANUP

gold_slim.unpersist()

# Restore original full-column view
gold.createOrReplaceTempView("integrated_taxi_trips")

Q1_monthly_demand_per_zone uncached: median=0.196s runs=[0.367, 0.189, 0.196]
Q2_avg_distance_by_weather uncached: median=0.212s runs=[0.244, 0.212, 0.082]
Q3_air_quality_vs_demand uncached: median=0.149s runs=[0.181, 0.149, 0.06]
Q4_zones_most_weather_sensitive uncached: median=0.162s runs=[0.276, 0.162, 0.135]
Q5_peak_hour_per_day uncached: median=0.442s runs=[0.442, 0.362, 0.691]
Q6_monthly_trends uncached: median=0.165s runs=[0.182, 0.165, 0.078]

=== UNCACHED PLAN: Q1_monthly_demand_per_zone ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [is_valid#28, pickup_zone#29, pickup_year#41, pickup_month#42]
Batched: true
Location: InMemoryFileIndex [file:/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Pla

Q1_monthly_demand_per_zone cached: median=0.152s runs=[0.176, 0.152, 0.097]
Q2_avg_distance_by_weather cached: median=0.159s runs=[0.182, 0.159, 0.076]
Q3_air_quality_vs_demand cached: median=0.078s runs=[0.114, 0.078, 0.056]
Q4_zones_most_weather_sensitive cached: median=0.261s runs=[0.289, 0.261, 0.162]
Q5_peak_hour_per_day cached: median=0.363s runs=[0.438, 0.363, 0.271]
Q6_monthly_trends cached: median=0.133s runs=[0.133, 0.179, 0.072]

=== CACHING BENCHMARK SUMMARY ===
Q1_monthly_demand_per_zone: uncached=0.196s  cached=0.152s  speedup=1.29x  same_result=True
Q2_avg_distance_by_weather: uncached=0.212s  cached=0.159s  speedup=1.33x  same_result=True
Q3_air_quality_vs_demand: uncached=0.149s  cached=0.078s  speedup=1.92x  same_result=True
Q4_zones_most_weather_sensitive: uncached=0.162s  cached=0.261s  speedup=0.62x  same_result=True
Q5_peak_hour_per_day: uncached=0.442s  cached=0.363s  speedup=1.22x  same_result=True
Q6_monthly_trends: uncached=0.165s  cached=0.133s  speedup=1.25x

In [4]:
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Application ID:", spark.sparkContext.applicationId)

Spark UI: http://172.27.138.95:4040
Application ID: local-1789576927632


In [2]:
spark.stop()